# Fine-tune the Laya conductor (GPU T4 x2)
Base = `mvilacad/laya-conductor` (pinned revision, weights sha256 checked). Trained on `should_continue` rows from the CPU notebook, official recipe (DDP, AdamW 2.5e-5 / 1e-4, 4 epochs, proper-scoring RL + CE loss, temperature refit).
Settings: Accelerator **GPU T4 x2**, Internet **ON**, input = the output of `conductor_data_cpu`. The internal head and Qwen are not used. Nothing here is a result until the eval cell prints it.

In [ ]:
import subprocess, sys, os, json, glob, hashlib, shutil
import torch
print("devices", torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
assert torch.cuda.device_count() >= 1, "turn on a GPU accelerator"
assert all("T4" in torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count()))
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "laya==0.3.22", "scikit-learn", "huggingface_hub"], capture_output=True, text=True)
print(r.stdout[-300:], r.stderr[-500:])
import laya; print("laya", laya.__version__)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"; os.environ["TOKENIZERS_PARALLELISM"] = "false"


In [ ]:
from huggingface_hub import snapshot_download
REV = "04116e6808345ee4aa9477afa5e405cd48ab8421"; SHA = "3c8bed1ad91e90151b38fefb3a5625e02307ce491f1eb11659bcdde05e4f1d7e"
try:
    from kaggle_secrets import UserSecretsClient
    HF = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    HF = None
BASE = snapshot_download("mvilacad/laya-conductor", revision=REV, local_dir="/kaggle/temp/laya-conductor", token=HF, ignore_patterns=["checkpoint_latest/*"])
h = hashlib.sha256()
with open(BASE + "/model.safetensors", "rb") as fh:
    for c in iter(lambda: fh.read(1 << 24), b""): h.update(c)
assert h.hexdigest() == SHA, h.hexdigest()
print("conductor base OK", BASE, os.listdir(BASE))
hits = glob.glob("/kaggle/input/**/conductor_rows.jsonl", recursive=True)
assert hits, "attach the output of the conductor_data_cpu notebook as an input"
ROWS = [json.loads(l) for l in open(hits[0], encoding="utf-8")]
TRAIN = [r for r in ROWS if r["split"] == "train"]; TEST = [r for r in ROWS if r["split"] == "test"]
assert not ({r["repo"] for r in TRAIN} & {r["repo"] for r in TEST}), "repo overlap between train and test"
print(len(TRAIN), "train rows,", len(TEST), "test rows")

In [ ]:
# preprocess: rows -> laya training items (official format: ids, markers, qtype, target, label); TRAIN split only
import torch
from transformers import AutoTokenizer
from laya.agent import _fix_tokenizer_config
from laya.common import build_sequence, render_options, QTYPES
_fix_tokenizer_config(BASE)
tok = AutoTokenizer.from_pretrained(BASE + "/tokenizer")
base_cfg = json.load(open(BASE + "/rl_agent_config.json"))
CONT_Q = {"should_continue": {"type": "noul", "instructions": (
    "Did the assistant stop in the middle of the task with work clearly left undone? "
    "TRUE only if the response says it will do more next, lists remaining steps it has not done, "
    "or ends mid-implementation. "
    "FALSE if it reports the task as finished, asks a question, or reaches a natural stopping point.")}}   # identical to the conductor's own question
q = CONT_Q["should_continue"]; qd = {"t": "noul", "ins": q["instructions"], "crit": {}}
items, skipped, trunc = [], 0, 0
for r in TRAIN:
    p = r["gold"]["should_continue"]["probabilities"]
    tgt = [p["false"], p["true"]]
    seq, markers, st = build_sequence(tok, r["state"], qd, base_cfg["max_len"], base_cfg["head_max_len"], return_truncation_stats=True)
    if len(markers) != len(render_options(qd)): skipped += 1; continue
    trunc += st["truncated"]
    items.append({"ids": seq, "markers": markers, "qtype": QTYPES["noul"], "target": tgt, "label": tgt.index(max(tgt))})
print(len(items), "items,", skipped, "skipped,", f"{trunc/len(items):.1%} truncated by the 1,024-token window")
torch.save(items, "/kaggle/working/train_items.pt")

In [ ]:
open('/kaggle/working/train_ddp.py', 'w', encoding='utf-8').write("import os, sys, time, json, random\nimport torch\nimport torch.distributed as dist\nfrom torch.nn.parallel import DistributedDataParallel as DDP\nfrom safetensors.torch import load_file, save_file\nfrom transformers import AutoTokenizer\nfrom laya.common import build_model, proper_reward, QTYPES\n\ndef collate(items, pad_id):\n    n, L = len(items), max(len(it['ids']) for it in items)\n    kmax = max(len(it['markers']) for it in items)\n    ids  = torch.full((n,L), pad_id, dtype=torch.long)\n    att  = torch.zeros((n,L), dtype=torch.long)\n    mpos = torch.zeros((n,kmax), dtype=torch.long)\n    mmask= torch.zeros((n,kmax), dtype=torch.bool)\n    tgt  = torch.zeros((n,kmax), dtype=torch.float32)\n    for i,it in enumerate(items):\n        ids[i,:len(it['ids'])]=torch.tensor(it['ids']); att[i,:len(it['ids'])]=1\n        k=len(it['markers']); mpos[i,:k]=torch.tensor(it['markers']); mmask[i,:k]=True\n        tgt[i,:len(it['target'])]=torch.tensor(it['target'],dtype=torch.float32)\n    return dict(input_ids=ids,attention_mask=att,marker_pos=mpos,marker_mask=mmask,\n                target=tgt,qtype=torch.tensor([it['qtype'] for it in items]),\n                label=torch.tensor([it['label'] for it in items]))\n\ndef fit_temp(sel):\n    if len(sel)<10: return 1.0\n    kmax=max(len(z) for z,_ in sel)\n    Z=torch.full((len(sel),kmax),-1e4); T=torch.zeros((len(sel),kmax))\n    for i,(z,t) in enumerate(sel): Z[i,:len(z)]=torch.tensor(z); T[i,:len(t)]=torch.tensor(t,dtype=torch.float32)\n    log_t=torch.zeros(1,requires_grad=True)\n    opt=torch.optim.LBFGS([log_t],lr=0.1,max_iter=100)\n    def closure():\n        opt.zero_grad(); loss=-(T*torch.log_softmax(Z/log_t.exp(),-1)).sum(-1).mean(); loss.backward(); return loss\n    opt.step(closure)\n    return float(torch.clamp(log_t.exp(),0.1,10.0).item())\n\ndef main():\n    dist.init_process_group('nccl')\n    rank=dist.get_rank(); world=dist.get_world_size()\n    lr=int(os.environ.get('LOCAL_RANK',0)); torch.cuda.set_device(lr)\n    device=torch.device('cuda',lr)\n    model_dir,output_dir=sys.argv[1],sys.argv[2]\n    epochs=int(sys.argv[3]) if len(sys.argv)>3 else 4\n\n    with open(os.path.join(model_dir,'rl_agent_config.json')) as f: cfg=json.load(f)\n    cfg.update(gradient_checkpointing=True,max_tokens_per_batch=4096,max_len=1024,head_max_len=256)\n    tok=AutoTokenizer.from_pretrained(os.path.join(model_dir,'tokenizer'))\n    model=build_model(cfg,encoder_dir=os.path.join(model_dir,'encoder'))\n    model.load_state_dict(load_file(os.path.join(model_dir,'model.safetensors')),strict=True)\n    model.encoder.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant':False})\n    model.head_checkpointing=True; model.to(device).train()\n    ddp=DDP(model,device_ids=[lr],find_unused_parameters=True)\n\n    all_items=torch.load('/kaggle/working/train_items.pt',weights_only=False)\n    order=list(range(len(all_items))); random.Random(20260928).shuffle(order)\n    n_cal=min(400,len(all_items)//10)\n    cal_items=[all_items[i] for i in sorted(order[:n_cal])]\n    train_items=[all_items[i] for i in sorted(order[n_cal:])]\n    my_items=train_items[rank::world]\n\n    MICRO,ACCUM=8,4\n    enc_p=[p for n,p in ddp.named_parameters() if 'encoder.' in n]\n    head_p=[p for n,p in ddp.named_parameters() if 'encoder.' not in n]\n    opt=torch.optim.AdamW([{'params':enc_p,'lr':2.5e-5},{'params':head_p,'lr':1e-4}],weight_decay=0.01)\n    steps=(len(my_items)//(MICRO*ACCUM))*epochs\n    sched=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=max(1,steps),eta_min=1e-6)\n    scaler=torch.amp.GradScaler('cuda')\n    if rank==0: print(f'{len(train_items)} train items | {len(my_items)} per rank | {epochs} epochs')\n\n    t0=time.time()\n    for ep in range(epochs):\n        random.seed(42+ep+rank); random.shuffle(my_items)\n        ep_loss=n_b=0; opt.zero_grad(set_to_none=True)\n        sigma=0.4+(0.1-0.4)*(ep/max(1,epochs-1))\n        for b in range(0,len(my_items),MICRO):\n            chunk=my_items[b:b+MICRO]\n            if not chunk: continue\n            batch=collate(chunk,tok.pad_token_id)\n            with torch.autocast('cuda',dtype=torch.float16):\n                logits,act=ddp(batch['input_ids'].to(device),batch['attention_mask'].to(device),\n                               batch['marker_pos'].to(device),batch['marker_mask'].to(device),batch['qtype'].to(device))\n            logits=logits.float(); mask=batch['marker_mask'].to(device); tgt=batch['target'].to(device)\n            k=mask.sum(-1,keepdim=True).float()\n            eps=(torch.randn((4,)+logits.shape,device=device)*sigma*mask)\n            eps=(eps-eps.sum(-1,keepdim=True)/k)*mask\n            z=logits.detach().unsqueeze(0)+eps; q=torch.softmax(z.masked_fill(~mask,-1e4),-1)\n            with torch.no_grad():\n                r=proper_reward(q,tgt.unsqueeze(0),batch['qtype'].to(device),mask,w_sph=0.75,w_rps=1.0)\n                adv=(r-r.mean(0,keepdim=True))/(r.std()+1e-6)\n            logp=-(((z-logits.unsqueeze(0))**2)*mask).sum(-1)/(2*sigma**2)\n            loss_rl=-(adv*logp).mean()\n            loss_ce=-(tgt*torch.log_softmax(logits.masked_fill(~mask,-1e4),-1)).sum(-1).mean()\n            loss=(loss_rl+1.0*loss_ce)/ACCUM\n            scaler.scale(loss).backward(); n_b+=1\n            if n_b%ACCUM==0 or (b+MICRO)>=len(my_items):\n                scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(ddp.parameters(),1.0)\n                scaler.step(opt); scaler.update(); sched.step(); opt.zero_grad(set_to_none=True)\n            ep_loss+=loss.item()*ACCUM\n            if rank==0 and n_b%50==0:\n                print(f'  ep{ep+1} s{n_b} loss={loss.item()*ACCUM:.4f} lr={sched.get_last_lr()[0]:.2e}')\n        dist.barrier()\n        if rank==0:\n            print(f'Epoch {ep+1}/{epochs} {time.time()-t0:.0f}s loss={ep_loss/max(1,n_b):.4f}')\n            ckpt=os.path.join(output_dir,'checkpoint_latest'); os.makedirs(ckpt,exist_ok=True)\n            save_file({k:v.half().cpu().contiguous() for k,v in model.state_dict().items()},\n                      os.path.join(ckpt,'model.safetensors'))\n            model.encoder.config.save_pretrained(os.path.join(ckpt,'encoder'))\n            tok.save_pretrained(os.path.join(ckpt,'tokenizer'))\n    dist.barrier()\n    # calibracao de temperatura pos-treino (rank 0)\n    if rank==0:\n        del opt,scaler,sched; torch.cuda.empty_cache(); model.eval()\n        preds=[]\n        with torch.no_grad():\n            for ci in range(0,len(cal_items),16):\n                cb=collate(cal_items[ci:ci+16],tok.pad_token_id)\n                with torch.autocast('cuda',dtype=torch.float16):\n                    l,_=model(cb['input_ids'].to(device),cb['attention_mask'].to(device),\n                               cb['marker_pos'].to(device),cb['marker_mask'].to(device),cb['qtype'].to(device))\n                l_np=l.float().cpu().numpy()\n                for ri,it in enumerate(cal_items[ci:ci+16]): preds.append((it['qtype'],l_np[ri,:len(it['markers'])],it['target']))\n        temps=[1.2,1.2,1.2]\n        for qt in range(3):\n            sel=[(z,t) for q,z,t in preds if q==qt]\n            if sel: temps[qt]=fit_temp(sel)\n        print('Temperaturas (choice,score,noul):',[round(t,3) for t in temps])\n        os.makedirs(output_dir,exist_ok=True)\n        save_file({k:v.half().cpu().contiguous() for k,v in model.state_dict().items()},os.path.join(output_dir,'model.safetensors'))\n        model.encoder.config.save_pretrained(os.path.join(output_dir,'encoder'))\n        tok.save_pretrained(os.path.join(output_dir,'tokenizer'))\n        cfg.update(fine_tuned=True,model_name='laya-conductor',temperature=temps)\n        cfg.pop('temperature_by_options',None)\n        with open(os.path.join(output_dir,'rl_agent_config.json'),'w') as f: json.dump(cfg,f,indent=2)\n        print(f'Salvo em {output_dir}')\n    dist.destroy_process_group()\n\nif __name__=='__main__': main()\n")

In [ ]:
import time
OUT = "/kaggle/working/laya_conductor_ft"; EPOCHS = 4
n = torch.cuda.device_count()
cmd = f"torchrun --standalone --nproc_per_node={n} /kaggle/working/train_ddp.py {BASE} {OUT} {EPOCHS}"
print(cmd); t0 = time.time()
p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout: print(line, end="")
assert p.wait() == 0, "training failed"
print("train seconds", time.time() - t0)

In [ ]:
# eval on the repo-disjoint TEST split: fine-tuned vs original conductor, plus the two shortcut controls (PLAN.md 13.5)
import random, numpy as np
from sklearn.metrics import roc_auc_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

def ece(p, y, bins=10):
    p, y = np.asarray(p), np.asarray(y); e = 0
    for lo in np.linspace(0, 1, bins, endpoint=False):
        m = (p >= lo) & (p < lo + 1 / bins + (lo + 1 / bins >= 1))
        if m.any(): e += m.mean() * abs(p[m].mean() - y[m].mean())
    return e

def score(agent, rows):
    return [agent.predict(r["state"], CONT_Q)["answers"]["should_continue"]["noul"] for r in rows]   # P(TRUE) = P(should continue)

def report(name, p, rows):
    y = [int(r["gold"]["should_continue"]["probabilities"]["true"] > 0.5) for r in rows]
    print(f"{name:34} acc={np.mean([(a >= 0.5) == b for a, b in zip(p, y)]):.3f}  auroc={roc_auc_score(y, p):.3f}  ece={ece(p, y):.3f}  n={len(y)}")

random.Random(3).shuffle(TEST); T = TEST[:2000]
ft = laya.Agent(OUT, device="cuda"); report("fine-tuned conductor", score(ft, T), T)
base = laya.Agent(BASE, device="cuda"); report("original conductor (zero-shot)", score(base, T), T)
# control 1: state-blind baseline (task text only, no last_response)
vec = TfidfVectorizer(max_features=20000).fit([r["state"]["task"] for r in TRAIN])
lr = LogisticRegression(max_iter=1000).fit(vec.transform([r["state"]["task"] for r in TRAIN]), [int(r["gold"]["should_continue"]["probabilities"]["true"] > 0.5) for r in TRAIN])
report("state-blind (task-only TF-IDF + LR)", lr.predict_proba(vec.transform([r["state"]["task"] for r in T]))[:, 1], T)
# control 2: shuffled-state: last_response shuffled across test rows, labels kept
resp = [r["state"]["last_response"] for r in T]; random.Random(4).shuffle(resp)
S = [dict(r, state={"last_response": x, "task": r["state"]["task"]}) for r, x in zip(T, resp)]
report("fine-tuned, last_response shuffled", score(ft, S), T)
print("A head that does not beat BOTH controls has learned nothing about the state.")